# 🔥 fold3 (distributed) 학습 — Colab T4

YouTube 2개 영상(의성 + eEP8a2u5PbA)을 영상별로 솎아 중복 제거 후,
StratifiedKFold(shuffle)로 train/test에 무작위 분산 → fold 붕괴 없는 7개 모델 학습.

**실행 전:** 런타임 → 런타임 유형 변경 → **T4 GPU**
결과는 `model_save/fireimage_dist/`, `results/fireimage_dist/` 에 저장(기존 grouped 결과와 분리).
예상 시간: ~5~8시간 (internimage가 병목). 끊기면 다시 실행 시 완료분은 평가만 하고 이어감.

In [ ]:
# Cell 1: GPU 확인 + 코드 클론
import torch, os
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음 — T4로 변경!')
assert torch.cuda.is_available(), 'T4 GPU 런타임으로 변경 후 다시 실행'
!git clone https://github.com/yuntaewon812/fireimage_detection.git /content/fireimage_detection
%cd /content/fireimage_detection

In [ ]:
# Cell 2: 패키지 설치
!pip install timm einops transformers yt-dlp kaggle -q
print('설치 완료')

In [ ]:
# Cell 3: Kaggle 인증 (KGAT_ 토큰)
import os, getpass, re
raw = getpass.getpass('Kaggle API Token (KGAT_...): ')
token = re.sub(r'[^A-Za-z0-9_\-]', '', raw)
print('토큰 길이:', len(token), '| 시작:', token[:5])
os.environ['KAGGLE_API_TOKEN'] = token
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
open(os.path.expanduser('~/.kaggle/access_token'), 'w').write(token)
os.chmod(os.path.expanduser('~/.kaggle/access_token'), 0o600)
!kaggle datasets list --user yuntarwon

In [ ]:
# Cell 4: 데이터 다운로드 (video1 + 비youtube)
import os, glob, zipfile
BASE = '/content/fireimage_detection/data/fireimage'
def dl(dataset, dst):
    os.makedirs(dst, exist_ok=True)
    os.system(f'kaggle datasets download {dataset} -p {dst} --unzip')
    for _ in range(2):
        zips = glob.glob(f'{dst}/**/*.zip', recursive=True)
        if not zips: break
        for z in zips:
            try:
                with zipfile.ZipFile(z) as zf: zf.extractall(dst)
                os.remove(z)
            except Exception as e: print('unzip err', e)
dl('yuntarwon/fireimage-abnormal',         f'{BASE}/abnormal')
dl('yuntarwon/fireimage-abnormal-youtube', f'{BASE}/abnormal/youtube')
dl('yuntarwon/fireimage-normal',           f'{BASE}/normal')
imgs=('.jpg','.jpeg','.png','.bmp')
a=sum(1 for f in glob.glob(f'{BASE}/abnormal/**/*',recursive=True) if f.lower().endswith(imgs))
n=sum(1 for f in glob.glob(f'{BASE}/normal/**/*',recursive=True) if f.lower().endswith(imgs))
print(f'normal {n:,} / abnormal {a:,}')

In [ ]:
# Cell 5: video2(eEP8a2u5PbA) 프레임 추출 → youtube2 (2번째 영상 소스)
!python data/youtube_preprocessor.py --url "https://www.youtube.com/watch?v=eEP8a2u5PbA" \
    --subdir youtube2 --sample_every 8 --max_abnormal 600 --max_normal 300
!python data/youtube_preprocessor.py --stats

In [ ]:
# Cell 6: distributed 학습 실행 (7개 모델 × 3 fold)
%cd /content/fireimage_detection
!python main_distribute.py --class_name fireimage

In [ ]:
# Cell 7: 결과 확인
import pandas as pd
print(pd.read_csv('/content/fireimage_detection/results/fireimage_dist/metrics.csv').to_string())

In [ ]:
# Cell 8: 결과 다운로드 (세션 종료 전 필수)
import shutil
shutil.make_archive('/content/dist_model_save', 'zip', '/content/fireimage_detection', 'model_save/fireimage_dist')
shutil.make_archive('/content/dist_results', 'zip', '/content/fireimage_detection', 'results/fireimage_dist')
from google.colab import files
files.download('/content/dist_model_save.zip')
files.download('/content/dist_results.zip')